In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import (accuracy_score,f1_score,roc_auc_score,confusion_matrix,classification_report)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
!pip install xgboost
from xgboost import XGBClassifier


from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
TW = pd.read_csv(
    r"../classification/Twitter/Absolute_labeling/Twitter-Absolute-Sigma-500.data",
    sep=",",
    header=None
)

groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label")  # ??????? ???????

TW.columns = columns

TW.head(2)

In [ ]:
TH = pd.read_csv(
    "../classification/TomsHardware/Absolute_labeling/TomsHardware-Absolute-Sigma-500.data",
    sep=",",
    header=None
)


groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  # ??????? ???????

TH.columns = columns


prefixes = {col.split("_")[0] for col in TH.columns if "_" in col}


TH.head()

### Twitter

In [ ]:
X = TW.drop(columns=["label"])
y = TW["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### Random Forest baseline

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Random Forest baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

### Random Forest with class_weight

In [ ]:
rf = RandomForestClassifier(random_state=42,class_weight='balanced')
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Random Forest with class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

### Random Forest with Grid Search

In [ ]:
import pandas as pd

data = {
    "Model": [
        "Random Forest",
        "Random Forest with class_weight",
        "Random Forest with RandomizedSearch"
    ],
    "Accuracy": [0.9685, 0.9682, 0.9690],
    "Precision ": [0.9295, 0.9378, 0.9338],
    "Recall ": [0.9095, 0.8986, 0.9087],
    "F1-score": [0.9194, 0.9178, 0.9211],
    "ROC-AUC": [0.9914, 0.9916, 0.9762]
}

df_rf_tw = pd.DataFrame(data)
df_rf_tw

### TOm Harware

In [ ]:
X = TH.drop(columns=["label"])
y = TH["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### Random Forest baseline

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Random Forest baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

### Random Forest with class_weight

In [ ]:
rf = RandomForestClassifier(random_state=42,class_weight='balanced')
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Random Forest with class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

### Random Forest with class_weight

In [ ]:
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", 0.5]
}

grid = GridSearchCV(rf_base, param_grid, cv=5, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

best_rf = grid.best_estimator_


In [ ]:
print("Random Forest with ??????")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

In [ ]:
import pandas as pd

data = {
    "Model": [
        "Random Forest",
        "Random Forest with class_weight",
        "Random Forest with RandomizedSearch"
    ],
    "Accuracy": [0.9665, 0.9703, 0.9703],
    "Precision ": [0.9762, 0.9763, 0.9763],
    "Recall ": [0.9691, 0.9753, 0.9753],
    "F1-score ": [0.9726, 0.9758, 0.9758],
    "ROC-AUC": [0.9953, 0.9951, 0.9951]
}

df_rf_th = pd.DataFrame(data)
df_rf_th